Dependencies and Environment Setup

In [ ]:
# 1. Install necessary libraries
!pip install -q -U google-genai pymupdf

# 2. Imports
import os
import json
import fitz  # PyMuPDF
from google import genai
from google.genai import types
from google.colab import userdata, drive

# 3. Mount Google Drive for persistent storage
drive.mount('/content/drive')

# 4. Initialize Gemini Client
# Ensure you have added 'GEMINI_API_KEY' to your Colab Secrets (Left sidebar > Key icon)
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# 5. Define Storage Paths
BASE_PATH = "/content/drive/MyDrive/JurisMind_PageIndex"
RAW_PDF_DIR = f"{BASE_PATH}/raw_pdfs"
INDEX_DIR = f"{BASE_PATH}/indices"
CONTENT_DIR = f"{BASE_PATH}/content"

for path in [RAW_PDF_DIR, INDEX_DIR, CONTENT_DIR]:
    os.makedirs(path, exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.2/764.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
Mounted at /content/drive


The Global Document Catalog

In [ ]:
def generate_global_catalog():
    catalog = {}
    pdf_files = [f for f in os.listdir(RAW_PDF_DIR) if f.endswith('.pdf')]

    print(f"Found {len(pdf_files)} PDF(s). Generating Global Catalog...")

    for pdf in pdf_files:
        path = os.path.join(RAW_PDF_DIR, pdf)
        doc = fitz.open(path)

        # Get the first 2 pages for context (usually contains title/preface)
        intro_text = ""
        for i in range(min(2, len(doc))):
            intro_text += doc[i].get_text()

        prompt = f"Provide a 2-sentence technical summary of the legal scope of this document: {intro_text[:2000]}"
        response = client.models.generate_content(model="models/gemini-flash-latest", contents=prompt)

        catalog[pdf] = {
            "summary": response.text.strip(),
            "total_pages": len(doc)
        }
        print(f"  [✓] Cataloged: {pdf}")

    with open(f"{INDEX_DIR}/catalog.json", "w") as f:
        json.dump(catalog, f, indent=4)
    print("\nGlobal Catalog saved to Google Drive.")

# Run Cataloging
generate_global_catalog()

Found 5 PDF(s). Generating Global Catalog...
  [✓] Cataloged: Indian_Patent_Rules_2003__1_till2021.pdf
  [✓] Cataloged: tables_patentRules.pdf
  [✓] Cataloged: TradeMark_Rules.pdf
  [✓] Cataloged: TradeMark_Rules-75-93.pdf
  [✓] Cataloged: design-rules-2001.pdf

Global Catalog saved to Google Drive.


Tier 2 & 3 - Multimodal Page Ingestion

In [ ]:
from google.genai import types

safety_config = [
    types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
]

In [ ]:
import time

api_key = 'API_KEY'
client = genai.Client(api_key=api_key)

def ingest_document(pdf_filename):
    pdf_path = os.path.join(RAW_PDF_DIR, pdf_filename)
    doc_id = pdf_filename.replace(".pdf", "").lower().replace(" ", "_")

    # Storage setup
    index_file = f"{INDEX_DIR}/{doc_id}_index.json"
    doc_content_path = f"{CONTENT_DIR}/{doc_id}"
    os.makedirs(doc_content_path, exist_ok=True)

    doc_index = {}
    if os.path.exists(index_file):
        with open(index_file, "r") as f: doc_index = json.load(f)

    doc = fitz.open(pdf_path)
    print(f"Processing {pdf_filename}...")

    for i in range(len(doc)):
        page_num = i + 1
        md_file = f"{doc_content_path}/page_{page_num}.md"

        if os.path.exists(md_file) and str(page_num) in doc_index:
            continue

        print(f"  -> Processing Page {page_num}...")
        page = doc.load_page(i)

        # --- 1. GET VERBATIM TEXT (No AI needed, 100% accurate) ---
        raw_verbatim_text = page.get_text("text")

        # --- 2. GET VISUAL DESCRIPTIONS (AI for things text can't see) ---
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes()

        visual_prompt = """Look at this legal page image.
        1. If there is a FLOWCHART or DIAGRAM, describe its logic paths in detail.
        2. If there is a TABLE, reconstruct it into a Markdown table.
        3. Provide a 1-sentence SUMMARY of the page topic.
        If there are no visuals, just provide the summary."""

        try:
            response = client.models.generate_content(
                model="gemini-flash-latest",
                contents=[types.Part.from_bytes(data=img_bytes, mime_type="image/png"), visual_prompt],
                config=types.GenerateContentConfig(safety_settings=safety_config)
            )

            # Extract summary and visual descriptions
            ai_output = response.text if response.text else "No visuals detected."

            # --- 3. STITCH TIER 3 TOGETHER ---
            # We keep your exact text and APPEND the AI's visual analysis
            final_content = f"""# VERBATIM TEXT
{raw_verbatim_text}

# VISUAL ANALYSIS (AI GENERATED)
{ai_output}
"""
            # Extract summary for Tier 2 Index
            summary = ai_output.split("\n")[0].replace("SUMMARY:", "").strip()

            # Save Files
            with open(md_file, "w") as f: f.write(final_content)
            doc_index[str(page_num)] = summary
            with open(index_file, "w") as f: json.dump(doc_index, f, indent=4)

            time.sleep(5) # Protect rate limit

        except Exception as e:
            print(f"     [!] Error on page {page_num}: {e}")
            continue

    print(f"Ingestion complete for {pdf_filename}")

# Run Ingestion for all files in the catalog
with open(f"{INDEX_DIR}/catalog.json", "r") as f:
    cat = json.load(f)
    for filename in cat.keys():
        ingest_document(filename)

# Refining tier-2 using tier-3

In [ ]:
def refine_tier2_from_tier3(doc_id):
    index_file = f"{INDEX_DIR}/{doc_id}_index.json"
    content_folder = f"{CONTENT_DIR}/{doc_id}"

    with open(index_file, "r") as f:
        doc_index = json.load(f)

    print(f"Refining Index for {doc_id}...")

    for page_num, old_summary in doc_index.items():
        # Only refine if the summary is vague or short
        if "table" in old_summary.lower() or "reconstruction" in old_summary.lower() or len(old_summary) < 30:
            md_path = f"{content_folder}/page_{page_num}.md"

            if os.path.exists(md_path):
                with open(md_path, "r") as f:
                    full_content = f.read()

                # Ask Gemini to create a specific legal summary based on the FULL text
                prompt = f"""
                You are a Senior Patent Analyst. Your goal is to create a high-density, searchable
                index entry for a legal RAG system.
                Based on the following full page content (Verbatim Text + AI Visual Analysis),
                provide a precise 1-2 sentence legal summary that identifies exactly what Rules,
                Sections, or specific Fee categories are mentioned. It should cover every aspect of the content on this page.

                CONTENT:
                {full_content[:3000]}
                """

                try:
                    response = client.models.generate_content(model="gemini-flash-latest", contents=prompt)
                    new_summary = response.text.strip()
                    doc_index[page_num] = new_summary
                    print(f"  [✓] Page {page_num} refined: {new_summary[:50]}...")
                    time.sleep(1) # Faster because it's just text
                except:
                    continue

    # Save the updated index
    with open(index_file, "w") as f:
        json.dump(doc_index, f, indent=4)
    print(f"Refinement complete for {doc_id}")

# Run refinement for your main document
refine_tier2_from_tier3("tables_patentrules")

In [ ]:
import os
import json
import time

# Keep your existing client setup
api_key = 'API_KEY'
client = genai.Client(api_key=api_key)

def refine_specific_index(doc_id):
    index_file = f"{INDEX_DIR}/{doc_id}_index.json"
    content_folder = f"{CONTENT_DIR}/{doc_id}"
    progress_file = f"{INDEX_DIR}/{doc_id}_progress.json"

    if not os.path.exists(index_file):
        print(f"Error: Index file not found.")
        return

    with open(index_file, "r") as f:
        doc_index = json.load(f)

    completed_pages = []
    if os.path.exists(progress_file):
        with open(progress_file, "r") as f:
            completed_pages = json.load(f)

    print(f"\n--- Resumable High-Visibility Refinement: {doc_id} ---")

    updated_count = 0
    page_keys = sorted(doc_index.keys(), key=int)

    for page_num in page_keys:
        if str(page_num) in completed_pages:
            continue

        current_summary = doc_index[page_num]
        md_path = f"{content_folder}/page_{page_num}.md"
        if not os.path.exists(md_path): continue

        with open(md_path, "r") as f:
            full_content = f.read()

        success = False
        while not success:
            try:
                # STEP 1: CRITIC
                critic_prompt = f"""
        Analyze the relationship between the SUMMARY and the FULL_CONTENT of a legal document page.

        SUMMARY: "{current_summary}"
        FULL_CONTENT: {full_content[:3000]}

        Evaluate if the SUMMARY is 'HIGH_FIDELITY' or 'LOW_FIDELITY' based on:
        1. Does it miss specific Rule, Section, or Form identifiers present in the text?
        2. Is it generic (e.g., just mentions 'a table' or 'a diagram' without describing its specific topic)?
        3. Is it too brief to provide unique semantic context?

        Respond ONLY with 'PASS' if it is high-quality or 'FAIL' if it needs refinement.
        """
                gate = client.models.generate_content(model="gemini-flash-latest", contents=critic_prompt)
                status = gate.text.strip().upper()

                if "FAIL" in status:
                    print(f"  Page {page_num}: FAIL -> REFINING...")
                    # STEP 2: REFINER
                    refiner_prompt = f"""
                      You are a Senior Legal Metadata Engineer. Rewrite this summary to be 'High-Density'.

                      CONTENT: {full_content[:4000]}

                      REQUIREMENTS:
                      1. Capture ALL Rule, Section, and Form identifiers.
                      2. Summarize specific scope of tables/visuals (e.g., 'Fee schedule for Form 1').
                      3. Use 1-2 noun-heavy, searchable sentences.
                      4. NO conversational filler (e.g., 'This page discusses').
                      """
                    response = client.models.generate_content(model="gemini-flash-latest", contents=refiner_prompt)
                    doc_index[page_num] = response.text.strip()
                    updated_count += 1
                else:
                    print(f"  Page {page_num}: PASS")

                success = True

                # --- CRITICAL SYNC: Save BOTH files immediately ---
                # This ensures if it crashes now, the data AND progress are safe.
                with open(index_file, "w") as f:
                    json.dump(doc_index, f, indent=4)

                completed_pages.append(str(page_num))
                with open(progress_file, "w") as f:
                    json.dump(completed_pages, f)

                time.sleep(5)

            except Exception as e:
                if "429" in str(e):
                    print(f"      Rate Limit. Waiting 15s...")
                    time.sleep(15)
                else:
                    print(f"      Error: {e}. Retrying in 15s...")
                    time.sleep(15)

    # Cleanup progress file only when 100% complete
    if os.path.exists(progress_file):
        os.remove(progress_file)

    print(f"\nFINISHED: {doc_id}. Refined {updated_count} pages.")

In [ ]:
refine_specific_index("trademark_rules")

# Refining tier-1

In [ ]:
import os
import json
import time

api_key = 'API_KEY'
client = genai.Client(api_key=api_key)

def regenerate_global_catalog_v2():
    catalog_path = f"{INDEX_DIR}/catalog.json"

    # Load existing catalog if it exists so we can resume
    if os.path.exists(catalog_path):
        with open(catalog_path, "r") as f:
            catalog = json.load(f)
    else:
        catalog = {}

    index_files = [f for f in os.listdir(INDEX_DIR) if f.endswith('_index.json')]
    print(f"Syncing Global Catalog for {len(index_files)} documents...")

    for idx_file in index_files:
        doc_id = idx_file.replace('_index.json', '')

        # RESUME LOGIC: Skip if this doc is already cataloged (unless you want to force redo)
        if doc_id in catalog and "routing_metadata" in catalog[doc_id]:
            print(f"  Skipping {doc_id} (Already Cataloged)")
            continue

        path = os.path.join(INDEX_DIR, idx_file)
        with open(path, 'r') as f:
            page_summaries = json.load(f)

        # NO LIMIT: We send the entire index (every single refined page summary)
        all_summaries_text = "\n".join([f"Page {k}: {v}" for k, v in page_summaries.items()])

        prompt = f"""
        You are a Global Legal Librarian. I will provide you with a COMPLETE index of every page
        in a legal document ({doc_id}).

        INDEX DATA:
        {all_summaries_text}

        TASK:
        Create a 'Routing Profile' so the search agent knows exactly when to open this document.
        1. SCOPE: What specific legal areas/rules are covered?
        2. KEY IDENTIFIERS: List every Rule, Section, Form, and Schedule mentioned.
        3. VISUAL ASSETS: Does it contain Fee Tables, Flowcharts, or Timelines?

        Format as a technical, keyword-dense profile (4-5 sentences).
        """

        success = False
        while not success:
            try:
                response = client.models.generate_content(model="gemini-flash-latest", contents=prompt)

                catalog[doc_id] = {
                    "routing_metadata": response.text.strip(),
                    "total_pages": len(page_summaries),
                    "index_file": idx_file
                }

                # SAVE IMMEDIATELY (Atomic Save)
                with open(catalog_path, "w") as f:
                    json.dump(catalog, f, indent=4)

                print(f"  [✓] Global Map Updated for: {doc_id}")
                success = True
                time.sleep(10) # 10s delay between documents to prevent 429s

            except Exception as e:
                if "429" in str(e):
                    print(f"     Rate Limit for {doc_id}. Waiting 30s...")
                    time.sleep(30)
                else:
                    print(f"     Error: {e}")
                    break # Stop on non-rate errors

    print("\nMaster Catalog is now fully synchronized and High-Fidelity.")

# Run the fixed sync
regenerate_global_catalog_v2()

In [ ]:
import os
import json
import time
from google import genai # Assuming the library setup

api_key = 'API_KEY'
client = genai.Client(api_key=api_key)

def regenerate_global_catalog_v2(force_redo=False):
    # INDEX_DIR = "indices" # Ensure this matches your path
    catalog_path = f"{INDEX_DIR}/catalog.json"

    if os.path.exists(catalog_path):
        with open(catalog_path, "r") as f:
            catalog = json.load(f)
    else:
        catalog = {}

    index_files = [f for f in os.listdir(INDEX_DIR) if f.endswith('_index.json')]
    print(f"Syncing Global Catalog for {len(index_files)} documents...")

    for idx_file in index_files:
        doc_id = idx_file.replace('_index.json', '')

        # RESUME LOGIC: Skip unless force_redo is True
        if not force_redo and doc_id in catalog and "routing_metadata" in catalog[doc_id]:
            print(f"   Skipping {doc_id} (Already Cataloged)")
            continue

        path = os.path.join(INDEX_DIR, idx_file)
        with open(path, 'r') as f:
            page_summaries = json.load(f)

        all_summaries_text = "\n".join([f"Page {k}: {v}" for k, v in page_summaries.items()])

        # --- UNIVERSAL ANTI-BIAS PROMPT ---
        prompt = f"""
        You are a Professional Information Architect and Librarian.
        I am providing an index of page-level summaries for the document: {doc_id}.

        INDEX DATA:
        {all_summaries_text}

        TASK:
        Generate a 'Differentiated Routing Profile' for this document.
        Your goal is to ensure a search agent can distinguish this document from others that might
        share similar terminology or structure.

        REQUIRED STRUCTURE:
        1. CATEGORICAL IDENTITY: Define the primary, specific subject matter. Use a "DOMAIN: [Subject]" prefix.
        2. BOUNDARY SETTING (EXCLUSIONS): Identify related topics that this document DOES NOT cover to
           prevent false-positive matches. (e.g., 'Focuses on X, excludes Y and Z').
        3. CONTEXTUAL IDENTIFIERS: List key Sections, Rules, or unique codes mentioned.
           CRITICAL: Prefix each identifier with a short version of the document name to make it unique.
        4. TECHNICAL ANCHORS: Extract 5-7 highly specific technical terms or jargon unique to this
           specific document's context. Avoid generic words like "rules" or "table."
        5. TARGET TRIGGERS: Describe the exact scenario or specific question type that should
           activate this document over any other.

        Format this as a single, high-density technical paragraph. Do not use conversational filler.
        """

        success = False
        while not success:
            try:
                # Using the appropriate model name
                response = client.models.generate_content(model="gemini-flash-latest", contents=prompt)

                # Validation: Ensure the model didn't fail
                if not response.text:
                    raise ValueError("Empty response from AI")

                catalog[doc_id] = {
                    "routing_metadata": response.text.strip(),
                    "total_pages": len(page_summaries),
                    "index_file": idx_file
                }

                with open(catalog_path, "w") as f:
                    json.dump(catalog, f, indent=4)

                print(f"  [✓] High-Fidelity Map Updated for: {doc_id}")
                success = True
                time.sleep(5) # Reduced sleep, but keep it to respect quotas

            except Exception as e:
                if "429" in str(e):
                    print(f"      Rate Limit. Waiting 30s...")
                    time.sleep(30)
                else:
                    print(f"      Error processing {doc_id}: {e}")
                    break

    print("\n Master Catalog is now Anti-Biased and High-Fidelity.")

# Note: Set force_redo=True to overwrite the old biased metadata
regenerate_global_catalog_v2(force_redo=False)

In [ ]:
import os
import json
import time

# --- CONFIGURATION ---
TARGET_ID = "trademark_rules"  # <--- CHANGE THIS to the failed doc ID
# INDEX_DIR = "indices"
catalog_path = f"{INDEX_DIR}/catalog.json"

def update_single_document(doc_id):
    # 1. Load the Index Data for the specific doc
    idx_file = f"{doc_id}_index.json"
    path = os.path.join(INDEX_DIR, idx_file)

    if not os.path.exists(path):
        print(f" Error: Could not find {path}")
        return

    with open(path, 'r') as f:
        page_summaries = json.load(f)

    all_summaries_text = "\n".join([f"Page {k}: {v}" for k, v in page_summaries.items()])

    # 2. Use the Universal Anti-Bias Prompt
    prompt = f"""
        You are a Professional Information Architect and Librarian.
        I am providing an index of page-level summaries for the document: {doc_id}.

        INDEX DATA:
        {all_summaries_text}

        TASK:
        Generate a 'Differentiated Routing Profile' for this document.
        Your goal is to ensure a search agent can distinguish this document from others that might
        share similar terminology or structure.

        REQUIRED STRUCTURE:
        1. CATEGORICAL IDENTITY: Define the primary, specific subject matter. Use a "DOMAIN: [Subject]" prefix.
        2. BOUNDARY SETTING (EXCLUSIONS): Identify related topics that this document DOES NOT cover to
           prevent false-positive matches. (e.g., 'Focuses on X, excludes Y and Z').
        3. CONTEXTUAL IDENTIFIERS: List key Sections, Rules, or unique codes mentioned.
           CRITICAL: Prefix each identifier with a short version of the document name to make it unique.
        4. TECHNICAL ANCHORS: Extract 5-7 highly specific technical terms or jargon unique to this
           specific document's context. Avoid generic words like "rules" or "table."
        5. TARGET TRIGGERS: Describe the exact scenario or specific question type that should
           activate this document over any other.

        Format this as a single, high-density technical paragraph. Do not use conversational filler.
        """

    # 3. Call the API
    try:
        print(f" Updating metadata for: {doc_id}...")
        response = client.models.generate_content(model="gemini-flash-latest", contents=prompt)

        # 4. Load current catalog, update just this entry, and save
        if os.path.exists(catalog_path):
            with open(catalog_path, "r") as f:
                catalog = json.load(f)
        else:
            catalog = {}

        catalog[doc_id] = {
            "routing_metadata": response.text.strip(),
            "total_pages": len(page_summaries),
            "index_file": idx_file
        }

        with open(catalog_path, "w") as f:
            json.dump(catalog, f, indent=4)

        print(f" Successfully updated {doc_id} in catalog.json")

    except Exception as e:
        print(f" Failed to update {doc_id}: {e}")

# Run the targeted update
update_single_document(TARGET_ID)

The Reasoning Agent (The Retrieval Step)

In [ ]:
import os
import json
import time

api_key = 'API_KEY'
client = genai.Client(api_key=api_key)

def ask_jurismind(query):
    print(f" Querying JurisMind: '{query}'")

    # --- STAGE 1: THE LIBRARIAN (Tier 1 Router) ---
    with open(f"{INDEX_DIR}/catalog.json", "r") as f:
        catalog = json.load(f)

    catalog_context = "\n".join([f"DOC_ID: {k} | {v['routing_metadata']}" for k, v in catalog.items()])

    # IMPROVED PROMPT: Instruct the model to respect DOMAINS and EXCLUSIONS
    router_prompt = f"""
    You are an expert Legal Librarian. Analyze the provided DOCUMENT metadata.

    DOCUMENTS:
    {catalog_context}

    QUERY: {query}

    TASK:
    1. Identify the PRIMARY DOMAIN of the query (Patent, Design, or Trademark).
    2. Check the EXCLUSIONS in the metadata. If a document explicitly 'Excludes' the query topic, REJECT IT.
    3. Match the query to the document with the most relevant TECHNICAL ANCHORS and TARGET TRIGGERS.

    Respond ONLY with the single DOC_ID.
    """

    try:
        doc_id = client.models.generate_content(model="gemini-flash-latest", contents=router_prompt).text.strip()
        # Add this line right after to clean the output:
        doc_id = doc_id.replace("DOC_ID:", "").strip()
        print(f" Librarian selected: {doc_id}")
    except Exception as e:
        print(f" Librarian Error: {e}")
        return

    # --- STAGE 2: THE NAVIGATOR (Tier 2 Page Selection) ---
    with open(f"{INDEX_DIR}/{doc_id}_index.json", "r") as f:
        doc_index = json.load(f)

    index_context = "\n".join([f"Page {k}: {v}" for k, v in doc_index.items()])

    # navigator_prompt = f"Identify page numbers for query. INDEX:\n{index_context[:15000]}\nQUERY: {query}\nRespond ONLY page numbers (e.g. 45, 46)."
    navigator_prompt = f"Identify page numbers for query. INDEX:\n{index_context}\nQUERY: {query}\nRespond ONLY page numbers (e.g. 45, 46)."

    try:
        page_resp = client.models.generate_content(model="gemini-flash-latest", contents=navigator_prompt).text.strip()
        target_pages = [p.strip() for p in page_resp.split(",") if p.strip().isdigit()]
        print(f" Navigator targeted pages: {target_pages}")
    except Exception as e:
        print(f" Navigator Error: {e}")
        return

    # --- STAGE 3: THE NEIGHBOR-AWARE FETCHER (Tier 3 Stitching) ---
    final_context_block = ""
    content_dir = f"{CONTENT_DIR}/{doc_id}"
    pages_to_fetch = set()
    for p in target_pages:
        n = int(p)
        pages_to_fetch.update([n-1, n, n+1])

    sorted_pages = sorted([p for p in pages_to_fetch if p > 0])
    print(f" Stitching Pages {sorted_pages}...")

    for p in sorted_pages:
        path = f"{content_dir}/page_{p}.md"
        if os.path.exists(path):
            with open(path, "r") as f:
                final_context_block += f"\n--- SOURCE: {doc_id} | PAGE {p} ---\n{f.read()}\n"

    # --- STAGE 4: THE RESILIENT REASONING AGENT ---
    # reasoner_prompt = f"""
    # You are 'JurisMind', a Senior Patent Attorney. Answer the query using ONLY the source content.
    # INSTRUCTIONS:
    # 1. Reassemble sentences split across page boundaries.
    # 2. Cite Rules and Page Numbers.
    # 3. If the data is in a table, explain it clearly.
    # SOURCE CONTENT:
    # {final_context_block}
    # QUERY: {query}
    # """

    reasoner_prompt = f"""
    You are 'JurisMind', a Senior Intellectual Property Attorney.
    Answer the query using ONLY the provided source content.

    INSTRUCTIONS:
    1. Reassemble sentences split across page boundaries.
    2. Cite the specific Act (e.g., Designs Act 2000 or Patents Act 1970) and Page Numbers[cite: 29, 35].
    3. If the query is about fees or costs, strictly use the tables provided in the source content[cite: 2, 217].
    4. If the source content does not contain the answer, state that you cannot find it in these specific rules.

    SOURCE CONTENT:
    {final_context_block}

    QUERY: {query}
    """

    print(" Reasoning Agent synthesizing answer...")

    success = False
    retries = 0
    while not success and retries < 5:
        try:
            response = client.models.generate_content(model="gemini-flash-latest", contents=reasoner_prompt)
            print("\n---  JURISMIND RESPONSE ---")
            print(response.text)
            print("----------------------------\n")
            success = True
        except Exception as e:
            if "503" in str(e) or "high demand" in str(e).lower():
                print(f"      Pro Model busy. Retrying in 10s... (Attempt {retries+1}/5)")
                time.sleep(10)
                retries += 1
            else:
                print(f"      Unexpected Error: {e}")
                break

In [ ]:
ask_jurismind("Does Indian Patent give protection worldwide?")

In [ ]:
ask_jurismind("Regarding the qualifying examination for patent agents, what are the specific marks required to be declared as having passed?")

In [ ]:
ask_jurismind("Regarding invention filings, what is the mandatory time limit for an applicant to furnish proof of the right to apply for a patent if it is not provided at the time of filing?")

In [ ]:
ask_jurismind("Is it necessary to visit the Indian Patent Office to transact any business relating to patent application?")

In [ ]:
ask_jurismind("Where can one find the information relating to published/ granted patent application?")

In [ ]:
ask_jurismind("According to Rule 2(e) of the Design Rules, 2001, what criteria must a 'Set' of articles meet?")

In [ ]:
ask_jurismind("Under Rule 3(2), if a document is submitted to the office via tele-fax or e-mail, within how many days must the original document be submitted?")

In [ ]:
ask_jurismind("According to Rule 5(2)(c), which of the following payment methods is strictly prohibited for the payment of fees? A. Demand Drafts, B. Cheques, C. Cash, D. Stamps and Indian postal orders")

In [ ]:
ask_jurismind("According to Rule 14(7), what is the minimum size for a representation of a design consisting of a repeating surface pattern?")

In [ ]:
ask_jurismind("Based on the First Schedule and Second Schedule, what is the prescribed fee for an application for restoration of a lapsed design?")

In [ ]:
ask_jurismind("What are the contents of the Patent office Journal?")

In [ ]:
ask_jurismind("Is there a fee for filing a representation opposing the grant of a patent?")

In [ ]:
ask_jurismind("Which form and section govern the compulsory licensing of a patent?")

In [ ]:
ask_jurismind("How are costs calculated for attending a hearing before the Controller?")

In [ ]:
ask_jurismind("How are costs calculated for attending a hearing before the Controller?")

In [ ]:
ask_jurismind("What are the requirements for maintaining a name in the register of patent agents?")

In [ ]:
ask_jurismind("Which form and section govern the compulsory licensing of a patent?")

In [ ]:
ask_jurismind("What is the standard fee for filing a trademark application for a single class?")

In [ ]:
ask_jurismind("How much does it cost to register Collective or Certification marks?")

In [ ]:
ask_jurismind("Is there an option for expedited services, and what do they cost?")

In [ ]:
ask_jurismind("Is there an option for expedited services related to trademarks, and what do they cost?")

In [ ]:
# check neighbour logic
# LLM be like, mujhe indices file dekhker kuch concrete nhi mil rha, so i shud do somehting else

def jurismind_search(query):
    # 1. Selection: Pick the Document (Tier 1)
    with open(f"{INDEX_DIR}/catalog.json", "r") as f:
        catalog = json.load(f)

    router_prompt = f"User Question: {query}\n\nDocument Catalog: {json.dumps(catalog)}\n\nIdentify the single most relevant PDF filename. Return only the filename."
    target_pdf = client.models.generate_content(model="models/gemini-flash-latest", contents=router_prompt).text.strip()
    doc_id = target_pdf.replace(".pdf", "").lower().replace(" ", "_")

    # 2. Selection: Pick the Page (Tier 2)
    with open(f"{INDEX_DIR}/{doc_id}_index.json", "r") as f:
        page_index = json.load(f)

    page_prompt = f"User Question: {query}\n\nPage Summaries for {target_pdf}:\n{json.dumps(page_index)}\n\nSelect the 1-2 most relevant page numbers. Return as a comma-separated list of numbers (e.g., 1, 45)."
    target_pages_raw = client.models.generate_content(model="models/gemini-flash-latest", contents=page_prompt).text.strip()
    target_pages = [p.strip() for p in target_pages_raw.split(",")]

    # 3. Retrieval: Load Tier 3 Markdown Content
    context = ""
    for p_num in target_pages:
        try:
            with open(f"{CONTENT_DIR}/{doc_id}/page_{p_num}.md", "r") as f:
                context += f"--- Content from Page {p_num} ---\n{f.read()}\n\n"
        except:
            continue

    # 4. Final Answer Generation
    answer_prompt = f"Using the following specific page context (which includes text and flowchart logic), answer the user query.\n\nContext:\n{context}\n\nQuery: {query}"
    final_answer = client.models.generate_content(model="gemini-1.5-pro", contents=answer_prompt).text

    return final_answer

# Example Usage:
print(jurismind_search("What is the timeline for First Examination Report response?"))

Ingestion driver code

In [ ]:
# --- JURISMIND INGESTION DRIVER ---

def run_full_ingestion():
    print("Starting JurisMind PageIndex Ingestion...")

    # 1. Check if raw_pdfs directory has files
    pdf_files = [f for f in os.listdir(RAW_PDF_DIR) if f.endswith('.pdf')]
    if not pdf_files:
        print(f"No PDF files found in {RAW_PDF_DIR}. Please upload your PDFs to that folder in Drive.")
        return

    # 2. Step 1: Generate/Update the Global Catalog (Tier 1)
    # This helps the agent know which document to pick later
    try:
        generate_global_catalog()
    except Exception as e:
        print(f"Error during Catalog generation: {e}")
        return

    # 3. Step 2: Process each PDF for Page Indexing and Content (Tier 2 & 3)
    # This is the multimodal step that handles your flowcharts and tables
    for pdf in pdf_files:
        print(f"\n--- Processing Document: {pdf} ---")
        try:
            ingest_document(pdf)
        except Exception as e:
            print(f"Failed to ingest {pdf}: {e}")
            continue

    print("\n SUCCESS: All Tiers are ready in Google Drive.")
    print(f"Metadata: {INDEX_DIR}")
    print(f"Markdown Content: {CONTENT_DIR}")

# Execute the driver
run_full_ingestion()